In [22]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
import operator
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
load_dotenv()

from typing import Literal
from pydantic import Field,BaseModel
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [23]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

In [24]:
class JOKE(TypedDict):
    topic:str
    joke:str
    explanation:str

In [25]:
def generate_joke(state:JOKE):
    prompt=f"generate a joke on the topic{state['topic']}"
    response=llm.invoke(prompt)
    return {"joke": response}   

In [26]:
def explain_joke(state:JOKE):
    prompt=f"explain the joke{state['joke']}"
    response=llm.invoke(prompt)
    return {"explanation": response}

In [43]:
checkpoint=InMemorySaver()


In [44]:
graph=StateGraph(JOKE)
graph.add_node("generate_joke",generate_joke)
graph.add_node("explain_joke",explain_joke)

graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","explain_joke")
graph.add_edge("explain_joke",END)

workflow=graph.compile(checkpointer=checkpoint)


In [45]:
config1={"configurable":{"thread_id":"1"}}
config2={"configurable":{"thread_id":"2"}}

In [50]:
result=workflow.invoke({"topic":"study"},config=config2)

In [51]:
result

{'topic': 'study',
 'joke': AIMessage(content='Why did the textbook go to therapy?\n\nBecause it had too many problems to study.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-08-16T10:11:21.3424291Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1462717900, 'load_duration': 242731400, 'prompt_eval_count': 32, 'prompt_eval_duration': 100117000, 'eval_count': 18, 'eval_duration': 1094629000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a00a0d-ecc5-76f3-9aef-66438da23de4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 18, 'total_tokens': 50}),
 'explanation': AIMessage(content='This is not a joke in the classical sense, but rather a play on words. Here\'s how it works:\n\nThe setup of the joke asks why a textbook went to therapy, implying that there might be some kind of emotional or psychological issue with the textbook.\n\nThe punchline "Because it had

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'gaming', 'joke': AIMessage(content='Why did the gamer bring a ladder to the party?\n\nBecause he wanted to level up his social game! (get it?)', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-08-16T10:10:21.1518203Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1933004900, 'load_duration': 262466100, 'prompt_eval_count': 33, 'prompt_eval_duration': 70867000, 'eval_count': 26, 'eval_duration': 1574238000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a00a0c-ffd0-7a41-b926-05ec85fd3ed9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 26, 'total_tokens': 59}), 'explanation': AIMessage(content='This is a joke in the form of a pun. Here\'s how it works:\n\n1. The setup: "Why did the gamer bring a ladder to the party?"\n2. The punchline: "Because he wanted to level up his social game!"\n3. The wordplay: In gaming, "leveling up" r

In [53]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'study', 'joke': AIMessage(content='Why did the textbook go to therapy?\n\nBecause it had too many problems to study.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-08-16T10:11:21.3424291Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1462717900, 'load_duration': 242731400, 'prompt_eval_count': 32, 'prompt_eval_duration': 100117000, 'eval_count': 18, 'eval_duration': 1094629000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a00a0d-ecc5-76f3-9aef-66438da23de4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 18, 'total_tokens': 50}), 'explanation': AIMessage(content='This is not a joke in the classical sense, but rather a play on words. Here\'s how it works:\n\nThe setup of the joke asks why a textbook went to therapy, implying that there might be some kind of emotional or psychological issue with the textbook.\n\nThe punchl

In [49]:
workflow.get_state_history(config1)

<generator object Pregel.get_state_history at 0x7710f0d05a60>

In [54]:
workflow.get_state_history(config2)

<generator object Pregel.get_state_history at 0x7710f0d05220>